# MNIST Classification with FabricPC

Converted from `mnist.py`. Run the cells in order using a Python environment with `fabricpc`, `jax`, and `optax` installed. The data loaders load MNIST; training runs for 20 epochs.


## Imports and JAX Setup


In [1]:
import jax
from fabricpc.nodes import Linear, IdentityNode
from fabricpc.core.topology import Edge
from fabricpc.graph_assembly import TaskMap, graph
from fabricpc.graph_initialization import initialize_params
from fabricpc.core.activations import TanhActivation, SoftmaxActivation
from fabricpc.core.energy import CrossEntropyEnergy
from fabricpc.core.inference import InferenceSGD
from fabricpc.core.initializers import XavierInitializer
import optax
from fabricpc.training import train, evaluate
from fabricpc.utils.data.dataloader import MnistLoader
from fabricpc import setup_jax

setup_jax()  # auto-detects the backend; pass platform="cuda" to require GPU
jax.config.update("jax_default_prng_impl", "threefry2x32")



/Users/yeabsira/Coursera-Deep-Learning-Specialization/.venv-fabric/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Define Nodes


In [2]:
pixels = IdentityNode(shape=(784,), name="pixels")
hidden1 = Linear(
    shape=(256,),
    activation=TanhActivation(),
    name="hidden1",
    weight_init=XavierInitializer(distribution="normal", gain=1.0),
)
hidden2 = Linear(
    shape=(64,),
    activation=TanhActivation(),
    name="hidden2",
    weight_init=XavierInitializer(distribution="normal", gain=1.0),
)
output = Linear(
    shape=(10,),
    activation=SoftmaxActivation(),
    energy=CrossEntropyEnergy(),  # Use cross-entropy for classification
    name="class",
    weight_init=XavierInitializer(distribution="normal", gain=1.0),
)



## Step 2: Build Graph


In [3]:
structure = graph(
    nodes=[pixels, hidden1, hidden2, output],
    edges=[
        Edge(source=pixels, target=hidden1.slot("in")),
        Edge(source=hidden1, target=hidden2.slot("in")),
        Edge(source=hidden2, target=output.slot("in")),
    ],
    task_map=TaskMap(x=pixels, y=output),  # x=input, y=target
    inference=InferenceSGD(eta_infer=0.05, infer_steps=20),
)



## Step 3: Initialize Parameters


In [4]:
# Matching seed values do not produce identical NumPy/JAX weights or shuffles.
master_rng_key = jax.random.PRNGKey(3)
graph_key, train_key, eval_key = jax.random.split(master_rng_key, 3)
params = initialize_params(structure, graph_key)



## Step 4: Train


In [5]:
train_config = {"num_epochs": 20}
batch_size = 128
optimizer = optax.sgd(learning_rate=0.1)

train_loader = MnistLoader(
    "train", batch_size=batch_size, tensor_format="flat", shuffle=True, seed=3,
    normalize_mean=0.0, normalize_std=1.0,
)
test_loader = MnistLoader(
    "test", batch_size=batch_size, tensor_format="flat", shuffle=False,
    normalize_mean=0.0, normalize_std=1.0,
)

result = train(
    params=params,
    structure=structure,
    train_loader=train_loader,
    optimizer=optimizer,
    config=train_config,
    rng_key=train_key,
    verbose=True,
)
trained_params = result.params



Epoch 2/20:   6%|▌         | 539/9380 [00:02<00:29, 297.08it/s, energy=0.1021, epoch=2/20]

Epoch 1/20 — energy: 0.2905, target_energy: 0.2376


Epoch 3/20:  11%|█         | 1009/9380 [00:03<00:21, 393.80it/s, energy=0.0559, epoch=3/20]

Epoch 2/20 — energy: 0.0982, target_energy: 0.0685


Epoch 4/20:  16%|█▌        | 1474/9380 [00:04<00:20, 379.95it/s, energy=0.0716, epoch=4/20]

Epoch 3/20 — energy: 0.0716, target_energy: 0.0473


... (30 outputs left out)

Epoch 20/20:  96%|█████████▌| 8985/9380 [00:22<00:00, 414.38it/s, energy=0.0205, epoch=20/20]

Epoch 19/20 — energy: 0.0179, target_energy: 0.0115


Epoch 20/20: 100%|██████████| 9380/9380 [00:23<00:00, 401.46it/s, energy=0.0161, epoch=20/20]


Epoch 20/20 — energy: 0.0172, target_energy: 0.0111


## Step 5: Evaluate with a forward pass, matching NumPy's predict()


In [6]:
# This option only selects evaluation behavior; training above still uses PC.
metrics = evaluate(
    trained_params, structure, test_loader, train_config, eval_key,
    algorithm="backprop",
)
print(f"Test Accuracy: {metrics['accuracy'] * 100:.2f}%")



Test Accuracy: 94.35%
